# Rag with SLMs

## Notebook Overview — Deep Learning Project 5 (RAG System and SLMs)

This Jupyter Notebook presents the implementation and evaluation of a **Retrieval-Augmented Generation (RAG)** pipeline using small language models, as required in Homework 5.

The notebook begins by loading the **WikiQA dataset**, which contains question–answer pairs grounded in Wikipedia sentences. The main objective is to design a system that can retrieve the most relevant documents and generate a final natural language response.

### 1. First Attempt

The notebook walks through the complete RAG architecture:

- Receiving a user question  
- Computing question embeddings using Transformer-based embedding models  
- Indexing candidate documents with **FAISS** for efficient retrieval  
- Retrieving the top-*k* most relevant sentences  
- Injecting retrieved context into a prompt  
- Generating an answer using a small language model  

### 2. Embedding Model Analysis

multiple embedding models will be compared (such as MiniLM variants), examining:

- The effect of embedding dimension  
- The quality of retrieved documents  
- Performance using **Recall@k**  

### 3. Small Language Model Comparison

A major section evaluates three different small language models:

- `flan-t5-small`
- `t5-small`
- `distilgpt2`

For each model, the notebook reports:

- Answer quality  
- Response time on CPU  
- Differences in style and reasoning behavior  

### 4. Lightweight Fine-Tuning (LoRA / PEFT)

After selecting the best-performing model, the notebook applies **parameter-efficient fine-tuning** using LoRA, training on a subset of WikiQA. The impact of fine-tuning on performance is discussed and re-evaluated.

### 5. Sensitivity Experiments

Finally, the notebook explores how key parameters affect the RAG system, including:

- Number of retrieved documents (*k*)  
- Prompt length  
- Generation temperature  

Results are presented, followed by conceptual discussion on:

- The importance of embedding quality  
- Challenges of running RAG fully on CPU  
- Possible improvements to the system  

---

Overall, this notebook serves as a complete experimental study of building and improving a CPU-based RAG assistant with small language models.


## Preparation

In this section:
- The required libraries for building the RAG pipeline are imported, including PyTorch, FAISS, transformers, sentence-transformers, and PEFT. 
- The execution environment is configured to suppress unnecessary warnings and ensure CPU-based execution, with an optional GPU check for training.
- Key constants are defined, including available embedding models and generation models.
- The WikiQA dataset is loaded, and the train, validation, and test splits are prepared for later retrieval and generation steps.


### Imports

In [3]:
import re
import os
import time
import torch
import faiss
import evaluate
import numpy as np
from pathlib import Path
from datasets import load_dataset
from __future__ import annotations
from collections import defaultdict
from transformers.utils import logging as hf_logging
from sentence_transformers import SentenceTransformer
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from typing import Callable, Dict, List, Optional, Union, Any, Sequence, Tuple
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,  TrainingArguments, Trainer, BitsAndBytesConfig

d:\Masters\Deep.Learning\Projects\HW5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar() 

### GPU check

This section is mainly focused on the device we are about to use and does not interfere with other parts and can be just replaced with the following code:
```python
device = torch.device("cpu")
```

In [5]:
!nvidia-smi

Mon Feb 16 23:14:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   33C    P0             19W /  128W |       0MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
test_device = device = torch.device("cpu")
if torch.cuda.is_available():
    train_device = torch.device("cuda")
    print("Using GPU for train:", torch.cuda.get_device_name(0))
else:
    train_device = torch.device("cpu")
    print("Using CPU for train")

Using GPU for train: NVIDIA GeForce RTX 4060 Laptop GPU


### Constants

In [7]:
EMBEDDING_MODELS = {
    'MiniLM': 'sentence-transformers/all-MiniLM-L6-v2',
    'SE5': 'intfloat/e5-small-v2',
    'E5': 'intfloat/e5-base-v2'
}
GENERATION_MODELS = {
    'ft5': ('google/flan-t5-small', 'seq2seq'),
    't5': ('google-t5/t5-small', 'seq2seq'),
    'gpt': ('distilbert/distilgpt2', 'causal')
}
PROMPTS = {
    'base': "Answer the following question using the provided context. If the answer is not clear, say \"I don't know\".\n\n"
            "Context:\n#Context\n\nQuestion: #Question\nAnswer:",
    'short': "Context:\n#Context\n\nQuestion: #Question\nThe Answer to the question considering context is:",
}

### Loading data

In [8]:
dataset = load_dataset("microsoft/wiki_qa")

train_ds = dataset['train']
print(len(train_ds))

validation_ds = dataset['validation']
print(len(validation_ds))

test_ds = dataset['test']
print(len(test_ds))

20360
2733
6165


In [9]:
all_docs = []
for split in dataset.keys():
    all_docs.extend(dataset[split]["answer"])

## Class Definition

### Text embedding

The `TextEmbedder` class is responsible for converting text into dense vector representations (embeddings) using a Transformer-based sentence embedding model. These embeddings are later used for semantic retrieval within the RAG pipeline.

#### Purpose in the Project

In this project, the class plays a central role in the **retrieval stage** of the RAG system. It is used to:

- Generate embeddings for all candidate answer sentences in the WikiQA dataset  
- Generate embeddings for incoming user questions  
- Enable similarity search using FAISS to retrieve the top-*k* most relevant documents  

The quality of these embeddings directly affects retrieval performance (e.g., Recall@k).

#### Methods and Capabilities

- `__init__(model_name, device)`
    - Loads a specified sentence embedding model  
    - Assigns the computation device (CPU or GPU)  
    - Initializes the embedding model for later use

- `embed(texts)`
    - Takes a list of input texts  
    - Encodes them into dense vector embeddings  
    - Returns embeddings as NumPy arrays  



In [10]:
class TextEmbedder:
    def __init__(self, model_name, device):
        self.device = device
        self.model = SentenceTransformer(model_name, device=self.device)
    
    def embed(self, texts):
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        return embeddings


### Document indexing

The `FaissIndex` class is responsible for building and querying a FAISS vector index to perform fast similarity search over embedded documents. It stores both the FAISS index (for nearest-neighbor search) and the original document texts so retrieved vector IDs can be mapped back to sentences during the retrieval stage.

#### Purpose in the Project

In this project, the class plays a central role in the **retrieval stage** of the RAG system. It is used to:

- Create a FAISS index for all embedded candidate sentences/documents
- Insert (index) precomputed embeddings along with their corresponding text
- Retrieve the top-*k* most relevant sentences for a given embedded question
- Return both similarity scores (distances) and the raw retrieved sentences for prompt/context building

This enables the system to perform efficient semantic retrieval before sending context to the generator model.

#### Methods and Capabilities

- `__init__(dim)`

  - Initializes a FAISS `IndexFlatL2` index with the given embedding dimension
  - Stores the expected embedding dimension for validation
  - Creates an internal list to store document texts aligned with FAISS vector IDs

- `add(embeddings, docs)`

  - Validates that embedding vectors match the expected dimension (`dim`)
  - Ensures the number of embeddings matches the number of documents
  - Adds embeddings into the FAISS index
  - Stores the corresponding raw documents/sentences in `self.documents`

- `search(query_vec, k)`

  - Reshapes the query vector into the format expected by FAISS
  - Retrieves the top-*k* nearest vectors using L2 distance
  - Extracts the retrieved vector IDs and distances
  - Maps IDs back to the original stored sentences
  - Returns `(ids, distances, sentences)` for downstream use in retrieval + prompting

In [11]:
class FaissIndex:
    def __init__(self, dim):
        self.index = faiss.IndexFlatL2(dim)
        self.dim = dim
        self.documents = []
    
    def add(self, embeddings, docs):
        assert embeddings.shape[1] == self.dim
        assert len(embeddings) == len(docs)
        self.index.add(embeddings)
        self.documents.extend(docs)
    
    def search(self, query_vec, k):
        D, I = self.index.search(query_vec.reshape(1, -1), k)
        ids = I[0]
        distances = D[0]
        sentences = [self.documents[i] for i in ids]
        return ids, distances, sentences


### Data retriever

The `Retriever` class is responsible for managing the end-to-end **embedding + indexing + retrieval** workflow in the RAG pipeline. It wraps a `TextEmbedder` to generate vectors, maintains a `FaissIndex` for similarity search, and tracks already-seen documents to avoid duplicate indexing.

#### Purpose in the Project

In this project, the class acts as the main **retrieval controller** used before generation. It is used to:

- Initialize the embedding model used for semantic search
- Build and maintain a FAISS index over candidate documents/sentences
- Prevent duplicate documents from being indexed multiple times
- Retrieve the top-*k* most relevant sentences for an input question
- Provide the retrieved sentences as context to the generator/model prompt

This class essentially connects our embedding model and FAISS index into a single reusable retrieval component.

#### Methods and Capabilities

- `__init__(embedder, device)`

  - Creates a `TextEmbedder` using a model name selected from `EMBEDDING_MODELS`
  - Stores a `seen` set to track and prevent duplicate documents
  - Computes the embedding dimension automatically by embedding an empty string
  - Initializes a `FaissIndex` using the computed embedding dimension

- `add_docs(documents)`

  - Filters the input documents to keep only **previously unseen** ones
  - Embeds only the unique new documents and converts embeddings to `float32` (FAISS-friendly)
  - Adds the embeddings + raw text into the FAISS index
  - Prints how many unique documents were added (or prints that nothing new exists)

- `retrieve(question, k)`

  - Embeds the input question into a query vector
  - Performs FAISS top-*k* similarity search against the indexed documents
  - Returns only the retrieved sentences (text strings) as the retrieval result

In [12]:
class Retriever:
    def __init__(self, embedder, device):
        self.embedder = TextEmbedder(EMBEDDING_MODELS[embedder], device)
        self.seen = set()
        self.embedding_dim = len(self.embedder.embed([''])[0])
        self.faiss = FaissIndex(dim=self.embedding_dim)
    
    def add_docs(self, documents):
        uq_documents = []
        for a in documents:
            if a not in self.seen:
                self.seen.add(a)
                uq_documents.append(a)
        if len(uq_documents) > 0:
            docs_embeddings = self.embedder.embed(uq_documents).astype("float32")
            self.faiss.add(docs_embeddings, uq_documents)
            print("Unique documents added:", len(uq_documents))
        else:
            print("There are no new information to be added.")

    def retrieve(self, question, k):
        q_vec = self.embedder.embed([question])[0]
        _, _, sentences = self.faiss.search(q_vec, k)
        return sentences

### Answer generator

The `Generator` class is responsible for producing the final natural-language answer in the RAG pipeline by loading a HuggingFace generation model (either **seq2seq** or **causal LM**), preparing prompts with a tokenizer, and running controlled text generation. It also optionally supports loading a **PEFT fine-tuned** seq2seq model.

#### Purpose in the Project

In this project, the class plays a central role in the **generation stage** of the RAG system. It is used to:

- Load the selected generation backbone from `GENERATION_MODELS`
- Tokenize the final prompt (typically question + retrieved context)
- Run the model’s `generate()` method with decoding parameters (e.g., max tokens, beam search)
- Decode generated tokens into a clean final answer string
- Optionally load a PEFT fine-tuned checkpoint (e.g., LoRA/PEFT output) for improved task performance

This is the final step after retrieval: it turns retrieved evidence + question into an answer.

#### Methods and Capabilities

- `__init__(model_key, device, ft_model=False, rank=8, alpha=16, bias='none')`

  - Selects the generation model name and type (`seq2seq` or `causal`) from `GENERATION_MODELS`
  - Loads the corresponding HuggingFace tokenizer (`AutoTokenizer`)
  - Ensures a valid padding token exists (falls back to EOS token if missing)
  - Loads the correct model class based on `model_type`:

    - `AutoModelForSeq2SeqLM` for seq2seq models
    - `AutoModelForCausalLM` for causal language models
  - If `ft_model=True` (seq2seq only):

    - Loads a PEFT fine-tuned checkpoint from a local path like
      `./PEFT/model/flan-t5-small-ft`
  - Moves the model to the specified compute device (CPU/GPU)

- `generate(prompt, **gen_kwargs)`

  - Tokenizes the input `prompt` into PyTorch tensors with truncation enabled
  - Runs `self.model.generate(...)` using any passed generation settings (`gen_kwargs`)
    (e.g., `max_new_tokens`, `num_beams`, `do_sample`, `temperature`)
  - Handles output extraction differently depending on model type:

    - For **causal** models, removes the prompt tokens from the output so only new tokens remain
    - For **seq2seq** models, uses the generated sequence directly
  - Decodes generated tokens into text, removes special tokens, trims whitespace
  - Returns the final answer string

In [13]:
class Generator:
    def __init__(self, model_key, device, ft_model=False, rank=8, alpha=16, bias='none'):
        self.model_name, self.model_type = GENERATION_MODELS[model_key]
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        if self.model_type == "seq2seq":
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
            if ft_model:
                self.model = PeftModel.from_pretrained(self.model, f"./PEFT/model/flan-t5-small-r{rank}-alpha{alpha}-bias-{bias}")
        elif self.model_type == "causal":
            self.model = AutoModelForCausalLM.from_pretrained(self.model_name)
        else:
            raise ValueError("model_type must be 'seq2seq' or 'causal'")
        self.device = device
        self.model = self.model.to(self.device)
    
    def generate(self, prompt, **gen_kwargs):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True).to(self.device)
        outputs = self.model.generate(**inputs, **gen_kwargs)
        if self.model_type == "causal":
            prompt_len = inputs["input_ids"].shape[1]
            generated = outputs[0][prompt_len:]
        else:
            generated = outputs[0]
        answer = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        return answer

### Retrieval augmented generation

The `RAG` class is responsible for orchestrating the full **Retrieval-Augmented Generation** workflow by combining a `Retriever` (to fetch relevant context) and a `Generator` (to produce the final answer). It builds a structured prompt from retrieved documents and controls generation behavior via temperature and decoding settings.

#### Purpose in the Project

In this project, the class represents the **end-to-end RAG pipeline controller**. It is used to:

- Retrieve the top-*k* most relevant sentences/documents for a user question
- Construct a unified prompt that includes both **context** and the **question**
- Enforce an “I don’t know” behavior when the context does not support an answer
- Call the generator model with sampling settings (temperature, repetition penalty)
- Return the final generated answer as the system output

This is the main interface that turns an input question into an answer using retrieval + generation.

#### Methods and Capabilities

- `__init__(retriever, generator, k, temp=0.5)`

  - Stores the provided `Retriever` instance for document lookup
  - Stores the provided `Generator` instance for text generation
  - Sets `max_docs` (`k`) to control how many retrieved documents are used as context
  - Sets `temperature` to control randomness/creativity during generation

- `build_prompt(question, docs)`

  - Limits the number of context documents to `max_docs` (if not `None`)
  - Joins retrieved documents into a single context block separated by newlines
  - Constructs an instruction-following prompt containing:

    - A rule to answer using context
    - A fallback instruction to respond with **"I don't know"** if unclear
    - The `Context:` section (retrieved docs)
    - The `Question:` section
    - An `Answer:` prefix to guide the model’s output
  - Returns the final prompt string passed to the generator

- `ask(question)`

  - Calls `self.retriever.retrieve(question, k=self.max_docs)` to fetch top-*k* docs
  - Builds the prompt using `build_prompt(question, docs)`
  - Calls `self.generator.generate(...)` with decoding controls:

    - `do_sample=True` (enables sampling instead of deterministic decoding)
    - `temperature=self.temperature` (controls sampling randomness)
    - `repetition_penalty=1.25` (reduces repeated text)
  - Returns the generated answer string as the final output

In [14]:
class RAG:
    def __init__(self, retriever, generator, k, temp=0.5, prompt_type='base'):
        self.retriever = retriever
        self.generator = generator
        self.max_docs = k
        self.temperature = temp
        self.base_prompt = PROMPTS[prompt_type]
    
    def build_prompt(self, question, docs):
        if self.max_docs is not None:
            docs = docs[:self.max_docs]
        context = "\n".join(docs)
        prompt = self.base_prompt.replace('#Context', context)
        return prompt.replace('#Question', question)
    
    def ask(self, question):
        docs = self.retriever.retrieve(question, k=self.max_docs)
        prompt = self.build_prompt(question, docs)
        answer = self.generator.generate(prompt, do_sample=True, temperature=self.temperature, repetition_penalty=1.25)
        return answer

## Evaluation

### LLM judge

The `LLMJudge` class implements an **LLM-as-a-judge** evaluator: it loads a small instruction-tuned language model (default: `Qwen/Qwen2-0.5B-Instruct`) and uses it to **score how semantically close a generated answer is to a reference answer**, returning a number from **0 to 100**.

#### Purpose in the Project

In RAG pipeline, we evaluate multiple small LMs (e.g., `flan-t5-small`, `t5-small`, `distilgpt2`) by checking the **quality of the final generated answer**. `LLMJudge` provides an *automatic* quality metric to compare models and settings (k, temperature, prompt length), complementing retrieval metrics like Recall@k. 

Concretely, it’s used in the **evaluation stage** after the RAG system produces an answer:

1. Get question → retrieve top-*k* docs → build prompt → generate answer
2. Compare generated answer vs. dataset reference answer using `LLMJudge.judge()`
3. Aggregate scores (mean/median) to compare configurations/models


#### What It Evaluates About an LLM? (and How?)

##### What it measures

`LLMJudge` measures **semantic similarity / conceptual alignment** between:

- **Reference** (ground-truth answer from dataset)
- **Generated answer** (our RAG system output)

It does *not* directly measure retrieval quality; it measures the *end-to-end* answer quality as judged by another model.

##### How it measures it

It constructs a strict scoring prompt:

- Tells the judge model to be “strict and impartial”
- Provides both texts (reference + generated)
- Instructs: *“Respond with a single integer between 0 and 100. Nothing else.”*

Then it:

- Tokenizes the prompt
- Calls `model.generate(...)` with **temperature = 0.0** and **do_sample = False** (deterministic)
- Decodes the output
- Extracts the last integer it can find via regex, clamps it to `[0, 100]`


#### Methods and Capabilities

- `__init__(model_name="Qwen/Qwen2-0.5B-Instruct")`

  - Loads a tokenizer and causal LM from HuggingFace
  - Uses `device_map="auto"` to place the model on available device(s)
  - Uses `float16` if CUDA is available, otherwise `float32`
  - Puts the model into inference mode with `eval()`

- `judge(reference, generated)`

  - Builds an evaluation prompt containing the reference and generated answer
  - Runs deterministic generation (`temperature=0.0`, `do_sample=False`)
  - Returns an integer score from `0` to `100` using `_parse_score`

- `_parse_score(text)` *(static)*

  - Finds any 1–3 digit integers in the model output
  - If none found → returns `0`
  - Otherwise takes the **last** matched integer and clamps it to `[0, 100]`


#### LLM Judge Ups and Downs
##### Strong Points

- **Fast, cheap automatic evaluation**: avoids manual grading and scales across many experiments (different models, k, temperature).
- **Semantic-aware scoring**: can reward answers that are correct but paraphrased (unlike exact-match metrics).
- **Deterministic outputs**: `temperature=0` + no sampling makes results stable run-to-run (useful for reporting).
- **Simple integration**: just needs `(reference, generated)`; easy to plug into our evaluation loop.


##### Weaknesses / Failure Modes

- **Judge bias & calibration**: the judge model’s notion of “closeness” may not match human judgment (or the task rubric). A 0.5B model can be especially inconsistent on nuanced cases.
- **Prompt sensitivity**: small changes in judge prompt wording can change scoring behavior significantly.
- **Reference dependence**: if the reference is short/incomplete (common in QA datasets), the judge may unfairly penalize valid answers that add correct details—or reward answers that match wording but miss correctness.
- **Can be fooled by fluent nonsense**: LLM judges sometimes over-score confident, well-written answers even if subtly wrong (hallucination-friendly scoring).
- **Parsing is brittle**:
  - If the model outputs multiple numbers (e.g., “Score: 85/100”), regex might still work, but if it outputs something weird (“eighty five”), score becomes `0`.
  - Taking the **last** number can backfire if the model includes extra digits later (example: “Score: 90\n(0-100 scale)” → last match might be 100).
- **Not a retrieval diagnostic**: a low score doesn’t tell us whether the problem is retrieval (bad docs) or generation (bad reasoning).


In [15]:
class LLMJudge:
    def __init__(self, model_name="Qwen/Qwen2-0.5B-Instruct"):
        print(f"Loading model: {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
        self.model.eval()
        print("Model ready.")

    def judge(self, reference, generated):
        prompt = (
            "You are a strict and impartial evaluator.\n\n"
            f"Reference:\n{reference}\n\n"
            f"Generated answer:\n{generated}\n\n"
            "Score how close is the meaning and the concept of the answer "
            "compared to the reference. "
            "Respond with a single integer between 0 and 100. Nothing else.\n"
            "Score:"
        )

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.0,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        decoded = self.tokenizer.decode(output[0], skip_special_tokens=True)
        return self._parse_score(decoded)

    @staticmethod
    def _parse_score(text):
        matches = re.findall(r"\b\d{1,3}\b", text)
        if not matches:
            return 0
        return max(0, min(100, int(matches[-1])))


### Google-BLEU

The `_normalize_references_for_multiref` + `evaluate_google_bleu` functions implement an **automatic n-gram overlap evaluator** using the HuggingFace `google_bleu` metric (often used as **GLEU-style BLEU**). Together, they **prepare references in the correct multi-reference format** and compute a single overlap-based score for our model’s generated answers against ground-truth answers.

#### Purpose in the Project

In RAG pipeline, we need to **evaluate the quality of final answers** produced by small LMs (e.g., `flan-t5-small`, `t5-small`, `distilgpt2`) and compare configurations (k, prompt length, temperature). `evaluate_google_bleu` provides a **text-similarity metric** that quantifies how much the generated answers **lexically match** the dataset references. 

Concretely, it’s used in the **evaluation stage** after the RAG system produces an answer:

1. Generate answers for a set of questions (RAG output)
2. Collect reference answers from the dataset
3. Compute `google_bleu` over `(predictions, references)` to compare models/settings

#### What It Evaluates About an LLM? (and How?)

##### What it measures?

These functions measure **surface-form similarity** between:

- **Prediction**: the LLM’s generated answer text
- **Reference(s)**: the ground-truth answer(s)

Specifically, **BLEU/GLEU-style** metrics measure **n-gram overlap** (how many 1-gram, 2-gram, 3-gram, 4-gram chunks match), which correlates with correctness when answers are expected to closely match the reference wording.

> This is **not** a semantic judge: it mostly rewards *wording overlap*, not conceptual equivalence.

##### How it measures it?

- `_normalize_references_for_multiref` converts references into the format required by the metric:

  - If references are single strings → wraps each reference into a list: `["ref"]`
  - If references are already lists (multi-ref) → strips each string

- `evaluate_google_bleu` then:
  - Verifies `len(predictions) == len(references)` (1 prediction per example)
  - Cleans predictions (handles `None`, strips whitespace)
  - Loads the metric via `evaluate.load("google_bleu")`
  - Computes overlap using the configured n-gram range (`min_len`..`max_len`)

#### Methods and Capabilities

- `_normalize_references_for_multiref(references)`

  - Handles **two reference formats**:

    - Single-reference per example: `["ref1", "ref2", ...]`
    - Multi-reference per example: `[["ref1a","ref1b"], ["ref2a"], ...]`
  - Normalizes both into **multi-ref format**:

    - `[[ref1], [ref2], ...]` or `[[ref1a, ref1b], [ref2a], ...]`
  - Strips whitespace and converts everything to string

- `evaluate_google_bleu(predictions, references, *, gleu_min_len=1, gleu_max_len=4)`

  - Validates that predictions and references align by index (same length)
  - Sanitizes predictions (`None` → `""`, strips whitespace)
  - Normalizes references to multi-ref format
  - Computes Google BLEU with an n-gram window we can control:

    - default: 1–4 grams (`gleu_min_len=1`, `gleu_max_len=4`)
  - Returns the metric result dict from `metric.compute(...)`

#### GLEU Ups and Downs
##### Strong Points

- **Simple, fast, and standard**: easy to run over many samples and compare models/settings quickly.
- **Reproducible**: deterministic scoring (no randomness like an LLM judge).
- **Good when wording matters**: works well if the dataset answers are short and expected to match reference phrasing.
- **Supports multi-reference answers**: `_normalize_references_for_multiref` lets us evaluate fairly when multiple correct phrasings exist.

##### Weaknesses / Failure Modes

- **Not semantic**: a correct paraphrase can score low if it uses different wording.
- **Penalizes verbosity**: longer answers may include extra tokens that reduce precision-style overlap.
- **Sensitive to phrasing/tokenization**: small wording differences (“NYC” vs “New York City”) can drop the score a lot.
- **Doesn’t detect hallucinations well**: a fluent but wrong answer can still share n-grams and score non-trivially.
- **Mismatch with RAG reality**: in RAG, answers may be *supported* by context but not match the exact reference string—BLEU may under-rate these correct answers.

In [16]:
def _normalize_references_for_multiref(references):
    if len(references) == 0:
        return []
    if isinstance(references[0], str):
        return [[str(r).strip()] for r in references]
    return [[str(r).strip() for r in ref_list] for ref_list in references]

In [17]:
def evaluate_google_bleu(predictions, references, *, gleu_min_len: int = 1, gleu_max_len: int = 4):

    if len(predictions) != len(references):
        raise ValueError(f"Length mismatch: {len(predictions)=} but {len(references)=}")

    preds = [("" if p is None else str(p)).strip() for p in predictions]
    refs_multi = _normalize_references_for_multiref(references)

    metric = evaluate.load("google_bleu")
    kwargs: Dict[str, Any] = {"min_len": gleu_min_len, "max_len": gleu_max_len}

    return metric.compute(predictions=preds, references=refs_multi, **kwargs)

## 1. RAG Deployment

In this section a simple flow has been deployed to show how the rag pipeline works. It includes creation of a retrieval system, a generator and a class that handles the functionality of these two.

In [16]:
ret = Retriever('SE5', test_device)

ret.add_docs(all_docs)

Unique documents added: 26196


In [17]:
gen = Generator('ft5', test_device)

In [18]:
rag = RAG(ret, gen, 5, 0.4)
print(rag.ask('HOW MUCH IS CENTAVOS IN MEXICO'))

100 centavos


## 2. Embedding Evaluation

The `build_groups` function organizes a dataset split (such as train, validation, or test) by grouping all entries that share the same `question_id`.

#### What It Does

For each example in the dataset:

- Uses `question_id` as the grouping key  
- Stores:
  - The **question text**
  - All candidate **answers**
  - Their corresponding **labels** (converted to integers)

In [19]:
def build_groups(ds_split, qid_col="question_id", question_col="question", answer_col="answer", label_col="label"):
    grouped = defaultdict(lambda: {"question": None, "answers": [], "labels": []})
    for ex in ds_split:
        qid = ex[qid_col]
        grouped[qid]["question"] = ex[question_col]
        grouped[qid]["answers"].append(ex[answer_col])
        grouped[qid]["labels"].append(int(ex[label_col]))
    return grouped

### Evaluation

This script is an **evaluation loop for a retriever** in a RAG-style pipeline. We compared multiple **embedding models** (from `EMBEDDING_MODELS`) on the **WikiQA validation split**, using **CPU**, and measured both:

* **Retrieval quality** (Precision@k, Recall@k)
* **Runtime costs** (model load time, document indexing time, evaluation time)



#### Code functionality

1. **Load and group the validation dataset**

   ```python
   validation_grouped = build_groups(validation_dataset)
   ```

   `build_groups` groups all rows by `question_id`, so each group contains:

   * one `question`
   * a list of candidate `answers`

2. **Loop over embedding models**
   For each `model_key`:

   * **Instantiate the retriever** and measure **load time**

     ```python
     retriever = Retriever(model_key, test_device)
     ```
   * **Index / upload all documents** into the retriever and measure **indexing time**

     ```python
     retriever.add_docs(all_docs)
     ```
   * **Evaluate retrieval at k=10**
     For each question:

     * retrieve `k` best matches: `best_matches = retriever.retrieve(question, k)`
     * count how many candidate answers appear in retrieved results (TP)

3. **Compute metrics**

   * **Precision@10 (as implemented here)**:
     [
     \frac{tp}{k \times #questions}
     ]
     This is: “out of all retrieved slots (10 per question), how many were correct matches?”
   * **Recall (as implemented here)**:
     [
     \frac{tp}{#all_candidates}
     ]
     This is: “out of all candidate answers in the validation set, how many were retrieved?”

> Note: This assumes `c in best_matches` is a valid correctness check (string match). That’s fine if `best_matches` are the same answer strings and normalization is consistent.


#### Results (k = 10, CPU)

| Embedding Model | Load Time (s) | Indexing / Upload Docs Time (s) | Eval Time (s) | Precision@10 | Recall |
| --------------- | ------------: | ------------------------------: | ------------: | -----------: | -----: |
| MiniLM          |         8.264 |                          61.254 |         2.782 |       38.18% | 41.35% |
| SE5             |        10.192 |                         116.255 |         4.526 |       32.97% | 35.71% |
| E5              |         7.553 |                         360.081 |         8.752 |       38.14% | 41.31% |



#### Analysis of the data

##### Accuracy (Precision / Recall)

* **MiniLM and E5 are basically tied** on quality:

  * Precision: **38.18% vs 38.14%**
  * Recall: **41.35% vs 41.31%**
* **SE5 is clearly worse**:

  * Precision down ~**5.2 points**
  * Recall down ~**5.6 points**

So in *retrieval accuracy*, the ranking is:
**MiniLM ≈ E5  >>  SE5**

##### Speed (CPU runtime)

* **Indexing time dominates** the total runtime cost.
* **E5 is extremely expensive to index on CPU**:

  * **360s** vs **61s** for MiniLM
  * That’s ~**5.9× slower** than MiniLM, for basically **no accuracy gain**.
* SE5 is in the middle but still slower than MiniLM:

  * **116s** indexing and worse accuracy.




#### Time–accuracy trade-off (and why MiniLM wins on CPU?)

On CPU, user usually care about:

* **embedding throughput** (how fast you can encode docs/questions)
* **online query latency** (how fast retrieval runs per question)

results show this typical trade-off:

* **E5** can be a very strong retriever in many setups, but **on CPU it can be costly**—especially when embedding tens of thousands of documents. Here, it gives **no meaningful accuracy improvement**, but costs **~6× more indexing time** and **~3× more eval time**.
* **MiniLM** achieves **the same retrieval quality** as E5 *in our pipeline* while being **much faster** to index and evaluate on CPU.
* **SE5** is slower than MiniLM *and* less accurate, so it’s not competitive in this experiment.

✅ **Conclusion:** For a CPU-based RAG system, **MiniLM is the best choice** in this experiment because it sits on the “sweet spot” of the trade-off: **near-best accuracy with much lower compute cost**, which matters a lot when user deploys without GPU or need frequent re-indexing.


In [20]:
validation_dataset = dataset['validation']
validation_grouped = build_groups(validation_dataset)
for model_key in EMBEDDING_MODELS:
    print()
    print(f"Embedding Model: {model_key}, Device: {device}")

    start = time.perf_counter()
    retriever = Retriever(model_key, test_device)
    end = time.perf_counter()
    print(f"Load-time: {end - start:.6f} seconds")

    start = time.perf_counter()
    retriever.add_docs(all_docs)
    end = time.perf_counter()
    print(f"Uploading docs-time: {end - start:.6f} seconds")

    k = 10
    print(f"Evaluation@{k}")
    start = time.perf_counter()
    positive = 0
    tp = 0
    for qid, item in validation_grouped.items():
        question = item["question"]
        candidates = item["answers"]
        best_matches = retriever.retrieve(question, k)
        positive += len(candidates)
        for c in candidates:
            if c in best_matches:
                tp += 1
    end = time.perf_counter()
    print(f"\tPrecision: {(tp / (k * len(validation_grouped))) * 100:.2f}%")
    print(f"\tRecall: {(tp / positive) * 100:.2f}%")
    print(f"\ttime: {end - start:.6f} seconds")
    print('#' * 80)



Embedding Model: MiniLM, Device: cpu
Load-time: 8.264395 seconds
Unique documents added: 26196
Uploading docs-time: 61.253789 seconds
Evaluation@10
	Precision: 38.18%
	Recall: 41.35%
	time: 2.782226 seconds
################################################################################

Embedding Model: SE5, Device: cpu
Load-time: 10.191761 seconds
Unique documents added: 26196
Uploading docs-time: 116.255314 seconds
Evaluation@10
	Precision: 32.97%
	Recall: 35.71%
	time: 4.525508 seconds
################################################################################

Embedding Model: E5, Device: cpu
Load-time: 7.553006 seconds
Unique documents added: 26196
Uploading docs-time: 360.081334 seconds
Evaluation@10
	Precision: 38.14%
	Recall: 41.31%
	time: 8.752169 seconds
################################################################################


#### Test experiment analysis

This experiment evaluates MiniLM on the test split using top-10 retrieval on CPU. The retriever loads in ~17.9s and indexes 5,951 unique answer documents in ~14.2s, which is significantly faster than indexing the larger validation corpus earlier (since the test set is smaller). Retrieval quality is noticeably stronger here, with Precision@10 = 50.24% and Recall = 51.58%, meaning that roughly half of the top-10 retrieved answers per question are relevant, and about half of all candidate answers are successfully retrieved. The evaluation loop itself takes ~5.76s, which is reasonable for CPU-based dense retrieval. Overall, these results indicate that MiniLM performs efficiently and achieves solid retrieval accuracy on the test set, maintaining a good balance between speed and effectiveness.

In [21]:
test_dataset = dataset['test']
test_grouped = build_groups(test_dataset)
print(f"Embedding Model: MiniLM, Device: {device}")

start = time.perf_counter()
retriever = Retriever('MiniLM', test_device)
end = time.perf_counter()
print(f"Load-time: {end - start:.6f} seconds")

start = time.perf_counter()
retriever.add_docs(list(test_dataset['answer']))
end = time.perf_counter()
print(f"Uploading docs-time: {end - start:.6f} seconds")

k = 10
print(f"Evaluation@{k}")
start = time.perf_counter()
positive = 0
tp = 0
for qid, item in test_grouped.items():
    question = item["question"]
    candidates = item["answers"]
    best_matches = retriever.retrieve(question, k)
    positive += len(candidates)
    for c in candidates:
        if c in best_matches:
            tp += 1
end = time.perf_counter()
print(f"\tPrecision: {(tp / (k * len(test_grouped))) * 100:.2f}%")
print(f"\tRecall: {(tp / positive) * 100:.2f}%")
print(f"\ttime: {end - start:.6f} seconds")

Embedding Model: MiniLM, Device: cpu
Load-time: 17.869929 seconds
Unique documents added: 5951
Uploading docs-time: 14.236139 seconds
Evaluation@10
	Precision: 50.24%
	Recall: 51.58%
	time: 5.755654 seconds


## 3. Generator Evaluation

### Grouping

This function groups a dataset by `question_id` and keeps **only questions that have a correct answer**.

#### What it does:
- Iterates over dataset samples.
- Stores question metadata (`question_id`, `question`, `document_title`).
- Keeps the answer where `label == 1` (correct answer).
- Filters out questions without a correct label.
- Returns a list of dictionaries in this format:

```python
{
    "question_id": ...,
    "question": ...,
    "document_title": ...,
    "correct_answer": ...
}

In [18]:
def question_answer_group(ds):
    q = {}
    correct = {}
    for ex in ds:
        qid = ex["question_id"]
        if qid not in q:
            q[qid] = {
                "question_id": qid,
                "question": ex["question"],
                "document_title": ex.get("document_title"),
            }

        if int(ex["label"]) == 1:
            correct.setdefault(qid, ex["answer"])

    out = []
    for qid, meta in q.items():
        if qid not in correct:
            continue

        out.append({
            **meta,
            "correct_answer": correct[qid],
        })

    return out

### Evaluation

This script evaluates a **RAG pipeline** on the *validation* split by comparing multiple **generation models** while keeping the **retriever fixed**.

#### How pipeline works
1. **Build a clean QA set**
   - `question_answer_group(validation_ds)` keeps only questions that have a single correct answer (`label == 1`).
   - Prints 5 reference (question → correct answer) examples.

2. **Initialize retrieval**
   - Uses a **MiniLM-based Retriever**.
   - Indexes `all_docs` into the retriever (`Unique documents added: 26196`).

3. **Initialize evaluation tools**
   - `LLMJudge()` (loaded as `Qwen/Qwen2-0.5B-Instruct`) scores predictions vs references on a **0–100** scale.
   - `evaluate_google_bleu(predictions, references)` computes **GLEU** using exact n-gram overlap.

4. **Loop through generation models**
   For each model in `GENERATION_MODELS`:
   - Load the generator and measure **load time**
   - Build a `RAG(retriever, generator, k=5, temp=0.1)`
   - For each QA item:
     - retrieve top-5 docs
     - generate an answer from retrieved context
   - Measure **generation time**
   - Compute **GLEU**
   - Compute **average LLMJudge score**



#### Output analysis

##### 1. Reference QA examples
These show the ground-truth answers are often **full factual sentences**, e.g.:
- “I Love Lucy ran from Oct 15, 1951 to May 6, 1957…”
- “Oklahoma bombing claimed 168 lives…”

So the evaluation expects answers that are:
- factually correct
- semantically aligned
- sometimes detailed (dates, numbers, names)

##### 2. Retriever indexing
`Unique documents added: 26196` confirms retrieval corpus is fairly large, so:
- generation quality depends heavily on whether the generator can *use retrieved context properly*.



#### Model-by-model performance

##### **ft5 (flan-t5-small/t5-small fine-tuned by FLAN)**
Sample outputs:
- BMC size → **“6,000”** (correct numeric extraction)
- I Love Lucy → **“six seasons”** (partially correct but not the full expected reference; still plausible)
- Pitbull fame → incomplete sentence (weakness)
- Owl family → **“Strigiformes”** (correct)
- Oklahoma bombing → **“168”** (correct)

Scores:
- **LLM Judge Avg: 61.58** (best)
- **GLEU: 0.0537** (low)

Interpretation:
- Often gives **short, precise answers** (good for factoid QA).
- Handles retrieval context reasonably well.
- Judge rewards semantic correctness even when phrasing differs.



##### **t5-small baseline**
Sample outputs:
- Frequently outputs the *instruction template itself*:
  - “Answer: How long was i love lucy on the air?”
  - “Answer the following question using the provided context…”
- This indicates **prompt / decoding failure** (model not following the task properly).

Scores:
- **LLM Judge Avg: 58.79** (second)
- **GLEU: 0.0961** (highest — but misleading)

Interpretation:
- Even though GLEU is highest, the samples show the model is often **not answering**.
- This can happen because overlap-based metrics may reward repeated tokens like “Answer”, “question”, or shared words from prompts/context.



##### **distilgpt2**
Sample outputs:
- Produces clearly wrong hallucinations:
  - “1 million units per year!”
  - “1 hour - 2 hours?”
  - “1 million - 2 million ... dead or wounded...”
- It’s not grounded in retrieved context and invents numbers.

Scores:
- **LLM Judge Avg: 54.19** (worst)
- **GLEU: 0.00818** (worst)

Interpretation:
- Hallucination-heavy and unreliable for factual QA here.
- Slowest generation time, too.



#### Comparing models

comparing *what the models actually said*:
- **ft5** answers with the right entities/numbers (“6000”, “Strigiformes”, “168”).
- **t5** often outputs the prompt template instead of an answer.
- **gpt** hallucinates dramatically.

And LLMJudge reflects that:
- **ft5 (61.58) > t5 (58.79) > gpt (54.19)**

✅ **Conclusion:** by *semantic correctness and grounding*, **flan-t5-small / ft5 is the best model** in our setup.



#### Why GLEU is not working?!

GLEU (like BLEU) is **n-gram overlap based**. In our task:
- The reference answers are often full sentences.
- Our best model (ft5) often returns **short factual spans** (e.g., “168”, “Strigiformes”).

That means:
- Even when ft5 is correct, it may share **few exact n-grams** with the longer reference → **low GLEU**.
- Meanwhile, a model that repeats the prompt (“Answer the following question…”) may accidentally share common tokens and inflate overlap → **higher GLEU**, despite being wrong.

So for QA with variable-length outputs and paraphrases, GLEU often **underestimates correct short answers** and **overestimates templated or noisy outputs**.



#### Runtime comparison table (CPU)

| Model Key | Load Time (s) | Generation Time (s) | 
|---|---:|---:|
| ft5 | 5.60 | **21.47** |
| t5 | **4.11** | 36.16 |
| gpt | 15.40 | 63.68 |



#### Why flan-t5-small is the best option on CPU

Considering **CPU constraints**, the best model should:
1. **Answer correctly** (highest quality / judge score)
2. **Run fast enough** (reasonable generation time)
3. **Be stable** (doesn’t output templates or hallucinate)

From the results:
- **ft5 is the most accurate** (best LLM judge + best sample correctness).
- **ft5 is also the fastest to generate** among the models that actually answer well.
- **t5 is slightly faster to load**, but generation is much slower and answers are often invalid templates.
- **gpt is both slow and inaccurate**.

✅ Therefore, for CPU-based RAG deployment, **flan-t5-small (ft5)** gives the best **time–accuracy trade-off** and is the most reliable generator in our experiment.


In [23]:
qa_grouped = question_answer_group(validation_ds)
q_count = len(qa_grouped)
print('Reference QA:')
for i in range(5):
    sample = qa_grouped[i]
    print(f"{sample['question']:50} ---> {sample['correct_answer']}")
retriever = Retriever('MiniLM', test_device)
retriever.add_docs(all_docs)
judge = LLMJudge()
for model_key in GENERATION_MODELS:
    print()
    print('#' * 100)
    print(f"Generating Model: {model_key}, Device: {device}")

    start = time.perf_counter()
    generator = Generator(model_key, test_device)
    end = time.perf_counter()
    print(f"Load-time: {end - start:.6f} seconds")

    rag = RAG(retriever, generator, 5, 0.1)
    predictions, references = [], []
    i = 0
    start = time.perf_counter()
    for item in qa_grouped:
        predictions.append(rag.ask(item['question']))
        references.append(item['correct_answer'])
        if i < 5:
            print(f"{item['question']:50} ---> {predictions[-1]}")
        i += 1
    end = time.perf_counter()
    print(f"Generation-time: {end - start:.6f} seconds")

    gleu_result = evaluate_google_bleu(predictions, references)
    print("GLEU", gleu_result)
    sum_judge = 0
    for i in range(len(references)):
        sum_judge += judge.judge(references[i], predictions[i])
    print(f"LLM Judge Avg score(0-100): {sum_judge / len(references)}")

Reference QA:
how big is bmc software in houston, tx             ---> Employing over 6,000, BMC is often credited with pioneering the BSM concept as a way to help better align IT operations with business needs.
how long was i love lucy on the air                ---> The black-and-white series originally ran from October 15, 1951, to May 6, 1957, on the Columbia Broadcasting System (CBS).
how did armando christian perez become famous      ---> Armando Pérez (born January 15, 1981), better known by his stage name Pitbull, is an American rapper, songwriter, and record producer.
what bird family is the owl                        ---> Owls are a group of birds that belong to the order Strigiformes, constituting 200 extant bird of prey species .
how many people were killed in the oklahoma city bombing ---> The Oklahoma blast claimed 168 lives, including 19 children under the age of 6, and injured more than 680 people.
Unique documents added: 26196
Loading model: Qwen/Qwen2-0.5B-Instruct ...


## 4. Fine Tunning & Ablation Study

### Fine tunning


#### What this code is doing

This notebook section performs **parameter-efficient fine-tuning (PEFT)** on **flan-t5-small** using **LoRA** to turn the model into a **binary classifier** that outputs either `"correct"` or `"incorrect"` for a *(question, candidate answer)* pair.

The training objective is **seq2seq text generation**, but the target text is only one token-level label: `"correct"` / `"incorrect"`.



#### Main components and how they work

##### 1. `fine_tune(...)` function
This function wraps the full LoRA training procedure:

- **Creates a `LoraConfig`**
  - `r`: LoRA rank (controls adapter capacity)
  - `lora_alpha`: scaling factor (controls effective update magnitude)
  - `target_modules=['q','v']`: LoRA is injected into the **query** and **value** projections in attention layers
  - `bias`: either `"lora_only"` or `"none"`
  - `task_type=SEQ_2_SEQ_LM`: correct for flan-t5-small

- **Wraps the base model with PEFT**
  - `model = get_peft_model(base_model, lora_config)`
  - Prints how many parameters are trainable (LoRA adapters only)

- **Defines Hugging Face training arguments**
  - batch size: 16 (train/eval)
  - learning rate: 1e-4
  - epochs: 3
  - evaluation each epoch (`eval_strategy='epoch'`)
  - generation enabled during evaluation (`predict_with_generate=True`)

- **Creates a `Seq2SeqTrainer`**
  - uses `DataCollatorForSeq2Seq`
  - trains on `train_tok`, evaluates on `val_tok`

- **Trains and saves LoRA adapters**
  - saves into a directory named with rank/alpha/bias



##### 2. `tokenize(batch)` function
This function turns raw dataset examples into seq2seq training inputs.

**Input text format** (per example):

```txt
Classify if the answer is correct.
Question:
Candidate Answer: ...
```

**Target text**:
- `"correct"` if label is 1
- `"incorrect"` otherwise

Tokenization details:
- inputs: max_length=512, truncation enabled
- targets: max_length=10
- labels are stored in `model_inputs["labels"]`



##### 3. Training loop over hyperparameters
The script fine-tunes multiple LoRA configurations:

- `rank ∈ {8, 16}`
- `alpha ∈ {16, 32}`
- `bias ∈ {"lora_only", "none"}`

For every combination:
- loads a fresh `AutoModelForSeq2SeqLM.from_pretrained(model_name)`
- calls `fine_tune(...)`

This ensures each run is independent and comparable.



#### Output analysis

##### Trainable parameter efficiency
Each run reports trainable parameters:

- **rank = 8**
  - trainable params: **344,064**
  - total params: **77,305,216**
  - trainable%: **0.445%**

- **rank = 16**
  - trainable params: **688,128**
  - total params: **~77.65M**
  - trainable%: **0.886%**

This shows LoRA updates a **tiny fraction** of the full model while still allowing meaningful learning.



##### Validation loss trend and best configurations
Across runs, validation loss consistently decreases over epochs, indicating stable learning.

**rank=8, alpha=16**
- eval_loss at epoch 3:
  - bias=lora_only: **0.09612**
  - bias=none: **0.09707**
- These two are very close, with a slight edge to `lora_only`.

**rank=8, alpha=32**
- eval_loss at epoch 3:
  - bias=lora_only: **0.08464**
  - bias=none: **0.08464**
- This is the **best validation loss** among all shown runs.

**rank=16, alpha=16**
- eval_loss at epoch 3:
  - ~**0.1004** (both biases)
- Worse than rank=8/alpha=32, suggesting extra capacity did not help here.

**rank=16, alpha=32**
- eval_loss at epoch 3:
  - ~**0.08501** (both biases)
- Very close to the best run, but still slightly higher than **0.08464**.

✅ Overall, the strongest configuration in these logs is:
- **rank=8, alpha=32** (bias does not materially change the outcome)



##### Training loss and convergence
Training loss mirrors the validation improvement:

- alpha=16 runs end around train_loss ≈ **0.198–0.200**
- alpha=32 runs end around train_loss ≈ **0.165–0.167**

This suggests increasing `alpha` improved optimization in this setup.



##### Runtime and throughput
Training time is similar across most runs:

- typical train_runtime: **~355–388 seconds**
- train_samples_per_second: **~157–172**
- train_steps_per_second: **~9.8–10.8**

Higher-rank / higher-alpha combinations sometimes reduce throughput slightly (e.g., rank=16 alpha=32 shows slower steps/sec), consistent with more adapter computation.



##### Bias setting effect
The `bias` option shows minimal impact:

- Most paired runs (`lora_only` vs `none`) produce nearly identical:
  - training loss
  - validation loss
  - runtime

So in this experiment, bias configuration appears **non-decisive** compared to rank/alpha.



##### Warnings during saving
Two repeated warnings appear:

- timeout fetching remote `config.json`
- PEFT assumes vocabulary was not modified

These warnings indicate a **network/HF Hub read timeout** during metadata lookup. They do **not** indicate training failure; the model still trains and saves adapters successfully.



#### Summary of what the output implies

From the logs, the experiment demonstrates that LoRA fine-tuning on flan-t5-small:
- trains only **0.45%–0.89%** of parameters
- converges reliably within **3 epochs**
- improves validation loss most strongly with **alpha=32**
- achieves best observed validation loss at **rank=8, alpha=32**
- shows negligible sensitivity to `bias` choice in this setting



In [38]:
def fine_tune(r, alpha, bias, base_model, tokenizer, train_tok, val_tok):
    lora_config = LoraConfig(
        r = r,
        lora_alpha = alpha,
        target_modules = ['q', 'v'],
        lora_dropout = 0.05,
        bias = bias,
        task_type = TaskType.SEQ_2_SEQ_LM
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    args = Seq2SeqTrainingArguments(
        output_dir= f"./PEFT/LOG/flan-t5-small-r{r}-alpha{alpha}-bias-{bias}",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=1e-4,
        num_train_epochs=3,
        logging_steps=500,
        eval_strategy='epoch',
        predict_with_generate=True,
        fp16=False,
        report_to="none",
    )
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=data_collator,
    )
    trainer.train()
    model.save_pretrained(f"./PEFT/model/flan-t5-small-r{r}-alpha{alpha}-bias-{bias}")

In [39]:
model_name, model_type = GENERATION_MODELS['ft5']
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [40]:
def tokenize(batch):
    inputs = [
        f"Classify if the answer is correct.\nQuestion: {q}\nCandidate Answer: {a}"
        for q, a in zip(batch['question'], batch['answer'])
    ]

    targets = ["correct" if label == 1 else "incorrect" for label in batch["label"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        targets,
        max_length=10,
        truncation=True,
        padding=False 
    )
    
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

In [41]:
train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
val_tok = validation_ds.map(tokenize, batched=True, remove_columns=validation_ds.column_names)

In [42]:
for rank in [8, 16]:
    for alpha in [16, 32]:
        for bias in ['lora_only', 'none']:
            print(f'\n\nFine-tunning with parameters: rank={rank}, alpha={alpha}, bias={bias}')
            base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
            fine_tune(rank, alpha, bias, base_model, tokenizer, train_tok, val_tok)



Fine-tunning with parameters: rank=8, alpha=16, bias=lora_only
trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451
{'loss': '0.7763', 'grad_norm': '0.1442', 'learning_rate': '8.693e-05', 'epoch': '0.3928'}
{'loss': '0.1281', 'grad_norm': '0.4928', 'learning_rate': '7.384e-05', 'epoch': '0.7855'}
{'eval_loss': '0.1218', 'eval_runtime': '5.09', 'eval_samples_per_second': '536.9', 'eval_steps_per_second': '33.59', 'epoch': '1'}
{'loss': '0.1109', 'grad_norm': '0.5922', 'learning_rate': '6.075e-05', 'epoch': '1.178'}
{'loss': '0.1161', 'grad_norm': '0.1835', 'learning_rate': '4.766e-05', 'epoch': '1.571'}
{'loss': '0.1163', 'grad_norm': '0.3259', 'learning_rate': '3.456e-05', 'epoch': '1.964'}
{'eval_loss': '0.1046', 'eval_runtime': '5.096', 'eval_samples_per_second': '536.4', 'eval_steps_per_second': '33.56', 'epoch': '2'}
{'loss': '0.1077', 'grad_norm': '0.1682', 'learning_rate': '2.147e-05', 'epoch': '2.357'}
{'loss': '0.1071', 'grad_norm': '0.4376', 'learning_ra

d:\Masters\Deep.Learning\Projects\HW5\.venv\Lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error The read operation timed out - silently ignoring the lookup for the file config.json in google/flan-t5-small.
  warnings.warn(
d:\Masters\Deep.Learning\Projects\HW5\.venv\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in google/flan-t5-small - will assume that the vocabulary was not modified.
  warnings.warn(




Fine-tunning with parameters: rank=8, alpha=32, bias=none
trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451
{'loss': '0.5701', 'grad_norm': '0.1563', 'learning_rate': '8.693e-05', 'epoch': '0.3928'}
{'loss': '0.1196', 'grad_norm': '0.6227', 'learning_rate': '7.384e-05', 'epoch': '0.7855'}
{'eval_loss': '0.1066', 'eval_runtime': '5.265', 'eval_samples_per_second': '519.1', 'eval_steps_per_second': '32.48', 'epoch': '1'}
{'loss': '0.1023', 'grad_norm': '0.9132', 'learning_rate': '6.075e-05', 'epoch': '1.178'}
{'loss': '0.1054', 'grad_norm': '0.2304', 'learning_rate': '4.766e-05', 'epoch': '1.571'}
{'loss': '0.1072', 'grad_norm': '0.5421', 'learning_rate': '3.456e-05', 'epoch': '1.964'}
{'eval_loss': '0.09032', 'eval_runtime': '5.245', 'eval_samples_per_second': '521.1', 'eval_steps_per_second': '32.6', 'epoch': '2'}
{'loss': '0.09922', 'grad_norm': '0.2353', 'learning_rate': '2.147e-05', 'epoch': '2.357'}
{'loss': '0.0975', 'grad_norm': '0.5743', 'learning_rate'

d:\Masters\Deep.Learning\Projects\HW5\.venv\Lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error The read operation timed out - silently ignoring the lookup for the file config.json in google/flan-t5-small.
  warnings.warn(
d:\Masters\Deep.Learning\Projects\HW5\.venv\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in google/flan-t5-small - will assume that the vocabulary was not modified.
  warnings.warn(


{'eval_loss': '0.09037', 'eval_runtime': '5.315', 'eval_samples_per_second': '514.2', 'eval_steps_per_second': '32.18', 'epoch': '2'}
{'loss': '0.1007', 'grad_norm': '0.1741', 'learning_rate': '2.147e-05', 'epoch': '2.357'}
{'loss': '0.09899', 'grad_norm': '0.3946', 'learning_rate': '8.379e-06', 'epoch': '2.749'}
{'eval_loss': '0.08501', 'eval_runtime': '6.797', 'eval_samples_per_second': '402.1', 'eval_steps_per_second': '25.16', 'epoch': '3'}
{'train_runtime': '388.4', 'train_samples_per_second': '157.2', 'train_steps_per_second': '9.832', 'train_loss': '0.1668', 'epoch': '3'}


Fine-tunning with parameters: rank=16, alpha=32, bias=none
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862
{'loss': '0.573', 'grad_norm': '0.1013', 'learning_rate': '8.693e-05', 'epoch': '0.3928'}
{'loss': '0.1212', 'grad_norm': '0.4209', 'learning_rate': '7.384e-05', 'epoch': '0.7855'}
{'eval_loss': '0.1094', 'eval_runtime': '5.286', 'eval_samples_per_second': '517.1', 'eval_steps_p

### Ablation study

This evaluation script tests multiple **LoRA fine-tuned flan-t5-small** checkpoints inside a **RAG pipeline**.

#### Steps performed
1. **Builds a reference QA set**
   - `question_answer_group(validation_ds)` keeps only questions that have a known `label==1` answer.
   - Prints 5 examples of *(question → correct_answer)* to show the ground truth style.

2. **Builds the retriever once**
   - A **MiniLM retriever** is created and indexed with `all_docs` (`26196` unique docs).
   - This retriever stays fixed so the only changing factor is the **generator checkpoint**.

3. **Loads an LLM-based evaluator**
   - `LLMJudge()` (Qwen2-0.5B-Instruct) scores each prediction against the reference on a **0–100** scale.

4. **Tests each LoRA configuration**
   - For every `(rank r, alpha, bias)` combination:
     - Loads the corresponding fine-tuned LoRA model using `Generator('ft5', ft_model=True, ...)`
     - Creates a RAG object: `RAG(retriever, generator, k=10, temp=0.1)`
     - Generates answers for every question in the grouped validation set
     - Computes:
       - **GLEU** (n-gram overlap)
       - **Average LLM Judge score** (semantic correctness)



#### Results summary

All configurations produce very similar output patterns:
- They consistently extract **“6,000”** for BMC (often correct).
- Many produce **“45 minutes”** for *I Love Lucy* (incorrect; the reference expects a multi-year run, not episode duration).
- Several models output **wrong or unrelated text** for the Pitbull question (e.g., “self-portraits”).
- Some runs correctly output **“168”** for the Oklahoma bombing, but others hallucinate **“1,833”**.

##### LLM Judge scores (higher is better)
| LoRA Config (r, α, bias) | LLM Judge Avg |
|---|---:|
| r=8, α=16, lora_only | **59.77** |
| r=8, α=16, none | 58.77 |
| r=8, α=32, lora_only | 56.05 |
| r=8, α=32, none | 57.33 |
| r=16, α=16, lora_only | 58.34 |
| r=16, α=16, none | 58.08 |
| r=16, α=32, lora_only | 56.52 |
| r=16, α=32, none | 57.16 |

##### Key observations from the data
- **α=16 generally outperforms α=32** on average judge score.
- **r=8 and r=16 are close**, but r=8 tends to be slightly stronger or comparable.
- **bias** has a relatively small effect overall, but `none` is often more stable (less extra parameters).



#### Impact of LoRA hyperparameters (r, α, bias)

##### 1. Rank `r`
- `r` controls the **capacity** of the LoRA adapters.
- Higher `r` means more trainable parameters and more expressive adaptation.
- However, if the task/data does not require extra capacity, higher `r` can:
  - add compute/memory cost
  - slightly increase instability or overfitting
- In these results, **r=16 does not improve quality over r=8**, implying the task likely does not benefit from extra adapter capacity.

##### 2. Alpha `α`
- `α` scales the LoRA update magnitude (effective strength of the adapter).
- Higher `α` increases how strongly LoRA affects the base model.
- In these results, **α=32 tends to reduce judge score**, suggesting that the LoRA updates may be too strong and can:
  - distort generation behavior
  - increase shallow patterns (e.g., repeating “45 minutes”, “nocturnal”)
- **α=16 behaves more controlled**, producing higher overall semantic scores.

##### 3. Bias mode
- `bias='none'`: bias terms are not trained/modified.
- `bias='lora_only'`: trains bias terms associated with LoRA layers.
- Bias tuning can help sometimes, but it also introduces extra degrees of freedom.
- Here, bias choice has **minor impact**, but `none` can be preferred for:
  - simplicity
  - slightly better generalization stability
  - fewer trainable parameters



#### Best configuration and why (third-person)

Among the tested configurations, **rank=8, alpha=16, bias='none'** is selected as the best practical option because:

- It achieves a **high LLM Judge score (58.77)** while remaining very close to the top score.
- It uses **lower LoRA capacity (r=8)** compared to r=16, making it more efficient and less prone to overfitting.
- It keeps **bias training disabled**, which reduces trainable complexity and often improves stability.
- It avoids the weaker behavior seen with **α=32**, where judge scores drop and outputs become less reliable.

In short, the results suggest that **a smaller adapter (r=8) with moderate scaling (α=16) and no bias tuning** provides the most balanced and CPU-friendly fine-tuned checkpoint for this RAG setup.

### The base model was already instruction-tuned

The LLM-judge score remains lower after fine-tuning because **flan-t5-small is already instruction-tuned on a large and diverse mixture of tasks**, including classification, QA, reasoning, and structured prompting. Its pretraining objective explicitly teaches it to follow natural language instructions and produce well-formed, context-aware answers. In contrast, the LoRA fine-tuning performed here narrows the model’s behavior to a very specific binary classification task (“correct” vs. “incorrect”). This specialization slightly shifts the model away from its broad instruction-following capabilities and generation style that originally made it strong in RAG answer synthesis. As a result, while the fine-tuned model may improve on the classification objective, it can degrade in open-ended QA generation quality, leading to lower LLM-judge scores compared to the original instruction-tuned base model.


In [46]:
qa_grouped = question_answer_group(validation_ds)
q_count = len(qa_grouped)
print('Reference QA:')
for i in range(5):
    sample = qa_grouped[i]
    print(f"{sample['question']:50} ---> {sample['correct_answer']}")
retriever = Retriever('MiniLM', test_device)
retriever.add_docs(all_docs)
judge = LLMJudge()
for rank in [8, 16]:
    for alpha in [16, 32]:
        for bias in ['lora_only', 'none']:
            print()
            print('#' * 100)
            print(f'Testing{GENERATION_MODELS["ft5"][0]} fine-tunned with rank={rank}, alpha={alpha}, bias={bias}:')
            generator = Generator('ft5', test_device, ft_model=True, rank=rank, alpha=alpha, bias=bias)

            rag = RAG(retriever, generator, 10, 0.1)
            predictions, references = [], []
            i = 0
            for item in qa_grouped:
                predictions.append(rag.ask(item['question']))
                references.append(item['correct_answer'])
                if i < 5:
                    print(f"{item['question']:50} ---> {predictions[-1]}")
                i += 1

            gleu_result = evaluate_google_bleu(predictions, references)
            print("GLEU", gleu_result)
            sum_judge = 0
            for i in range(len(references)):
                sum_judge += judge.judge(references[i], predictions[i])
            print(f"LLM Judge Avg score(0-100): {sum_judge / len(references)}")

Reference QA:
how big is bmc software in houston, tx             ---> Employing over 6,000, BMC is often credited with pioneering the BSM concept as a way to help better align IT operations with business needs.
how long was i love lucy on the air                ---> The black-and-white series originally ran from October 15, 1951, to May 6, 1957, on the Columbia Broadcasting System (CBS).
how did armando christian perez become famous      ---> Armando Pérez (born January 15, 1981), better known by his stage name Pitbull, is an American rapper, songwriter, and record producer.
what bird family is the owl                        ---> Owls are a group of birds that belong to the order Strigiformes, constituting 200 extant bird of prey species .
how many people were killed in the oklahoma city bombing ---> The Oklahoma blast claimed 168 lives, including 19 children under the age of 6, and injured more than 680 people.
Unique documents added: 26196
Loading model: Qwen/Qwen2-0.5B-Instruct ...


## 5. Hyper-parameter Sensitivity

### What this code does and how it works

This script runs a **hyper-parameter sweep** for a CPU-based **RAG (Retrieval-Augmented Generation)** system using:

- **Retriever:** MiniLM (fixed)
- **Generator:** a **LoRA fine-tuned flan-t5-small** checkpoint (`ft5`, r=8, α=16, bias=none)
- **Evaluation:**  
  - **GLEU** (n-gram overlap)
  - **LLMJudge average score (0–100)** using Qwen2-0.5B-Instruct

### Workflow (per experiment)
1. Builds a clean QA set from `validation_ds` using `question_answer_group`.
2. Indexes `all_docs` into the MiniLM retriever once.
3. Loads the fine-tuned generator once.
4. Loops over:
   - `k ∈ {5, 7, 10}` (how many docs retrieved)
   - `temp ∈ {0.1, 0.8, 1.6}` (sampling randomness)
   - `prompt_type ∈ {base, short}` (prompt template)
5. For each combination:
   - Creates a `RAG(retriever, generator, k, temp, prompt_type=...)`
   - Generates answers for all questions
   - Computes **GLEU** and **LLMJudge avg score**
   - Prints 5 sample predictions for qualitative comparison



### Impact of k, temperature, and prompt type

#### 1. `k` — number of retrieved documents
- **Lower k (5)**: less context, but often **cleaner** and reduces distraction.
- **Higher k (10)**: more context, but increases the chance of **noise** or conflicting facts.
- In the samples, increasing k sometimes introduces wrong numbers (e.g., *Oklahoma bombing → 1,833* at k=10, temp=0.1, base), suggesting **retrieval noise can mislead generation**.

#### 2. `temperature` — randomness in generation
- **temp = 0.1**: most deterministic and stable; outputs stay factual more often.
- **temp = 0.8**: more variation; sometimes still correct, but starts drifting.
- **temp = 1.6**: high randomness; outputs frequently hallucinate (e.g., 3380, 4280, 10488, 400000), and answers degrade sharply.
- The scores reflect this: **temp=1.6 consistently lowers LLMJudge and GLEU**.

#### 3. `prompt_type` — prompt formatting
- **base prompt**: longer instruction (“If not clear say I don’t know”) + explicit “Answer:” format.
- **short prompt**: more direct and simpler (“The Answer to the question considering context is:”).
- In these results, the **short prompt often improves judge score**, especially at low temperature, likely because it:
  - reduces instruction clutter
  - encourages direct extraction from context
  - reduces the chance the model repeats template-like text



### Results table

| k | temp | prompt | GLEU | LLM Judge Avg |
|---:|---:|---|---:|---:|
| 5 | 0.1 | base  | 0.0520 | 59.1587 |
| 5 | 0.1 | short | **0.0579** | **61.1508** |
| 5 | 0.8 | base  | 0.0376 | 55.8175 |
| 5 | 0.8 | short | 0.0472 | 59.8016 |
| 5 | 1.6 | base  | 0.0249 | 55.2698 |
| 5 | 1.6 | short | 0.0239 | 57.9127 |
| 7 | 0.1 | base  | 0.0520 | 58.3413 |
| 7 | 0.1 | short | 0.0587 | 60.2063 |
| 7 | 0.8 | base  | 0.0324 | 57.9206 |
| 7 | 0.8 | short | 0.0424 | 58.9365 |
| 7 | 1.6 | base  | 0.0156 | 55.2222 |
| 7 | 1.6 | short | 0.0265 | 56.8571 |
| 10 | 0.1 | base | 0.0479 | 58.2778 |
| 10 | 0.1 | short| 0.0557 | 59.6667 |
| 10 | 0.8 | base | 0.0368 | 59.5079 |
| 10 | 0.8 | short| 0.0459 | 58.6825 |
| 10 | 1.6 | base | 0.0213 | 55.1429 |
| 10 | 1.6 | short| 0.0126 | 57.0397 |



### Analysis of the results

#### Strongest pattern: low temperature wins
Across almost all k and prompt settings:
- **temp=0.1** yields the most consistent factual answers
- **temp=1.6** causes severe hallucination and score collapse

This matches the sample outputs:
- temp=0.1 frequently returns correct values like **“168”** and **“Strigiformes”**
- temp=1.6 returns nonsensical or wildly incorrect numbers

#### Prompt: short is generally better
At the best-performing temperatures (0.1), **short prompt > base prompt**:
- k=5: 61.15 (short) vs 59.16 (base)
- k=7: 60.21 (short) vs 58.34 (base)
- k=10: 59.67 (short) vs 58.28 (base)

Even in samples, short prompt often produces more direct answers (less template noise).

#### k: more retrieval is not always better
- Best score occurs at **k=5**, not 7 or 10.
- Increasing k sometimes injects misleading context:
  - Example: at **k=10, temp=0.1**, the model outputs **“45 minutes”** and **“1,833”**, which are incorrect compared to reference style.
- This suggests that for this dataset, the retriever’s top few results are sufficient, and extra documents add noise.



### Best hyper-parameters (chosen from the evidence)

**Best configuration:**  
✅ **k = 5, temperature = 0.1, prompt_type = short**

#### Why this is chosen
- It has the **highest LLMJudge score**: **61.15**
- It also has the **highest GLEU** among the runs: **0.0579**
- The **sample answers are the most stable and factual**:
  - “6,000”, “Strigiformes”, “168” appear correctly and consistently
- It avoids the two main failure modes observed elsewhere:
  - **high temperature hallucinations**
  - **higher k retrieval noise**

Therefore, based on both **quantitative scores** and **qualitative sample outputs**, the configuration *(k=5, temp=0.1, short prompt)* is the most reliable RAG setting for this fine-tuned generator on CPU.


In [53]:
qa_grouped = question_answer_group(validation_ds)
q_count = len(qa_grouped)
print('Reference QA:')
for i in range(5):
    sample = qa_grouped[i]
    print(f"{sample['question']:50} ---> {sample['correct_answer']}")
retriever = Retriever('MiniLM', test_device)
retriever.add_docs(all_docs)
judge = LLMJudge()
rank = 8
alpha = 16
bias = 'none'
generator = Generator('ft5', test_device, ft_model=True, rank=rank, alpha=alpha, bias=bias)
print(f'Testing{GENERATION_MODELS["ft5"][0]} fine-tunned with rank={rank}, alpha={alpha}, bias={bias}:')
for k in [5, 7, 10]:
    for temp in [0.1, 0.8, 1.6]:
        for prompt_type in PROMPTS:
            print()
            print('#' * 100)
            print(f'Hyper-parameters: k = {k}, temp = {temp}, prompt type = {prompt_type}')
            rag = RAG(retriever, generator, k, temp, prompt_type=prompt_type)
            predictions, references = [], []
            i = 0
            for item in qa_grouped:
                predictions.append(rag.ask(item['question']))
                references.append(item['correct_answer'])
                if i < 5:
                    print(f"{item['question']:50} ---> {predictions[-1]}")
                i += 1

            gleu_result = evaluate_google_bleu(predictions, references)
            print("GLEU", gleu_result)
            sum_judge = 0
            for i in range(len(references)):
                sum_judge += judge.judge(references[i], predictions[i])
            print(f"LLM Judge Avg score(0-100): {sum_judge / len(references)}")

Reference QA:
how big is bmc software in houston, tx             ---> Employing over 6,000, BMC is often credited with pioneering the BSM concept as a way to help better align IT operations with business needs.
how long was i love lucy on the air                ---> The black-and-white series originally ran from October 15, 1951, to May 6, 1957, on the Columbia Broadcasting System (CBS).
how did armando christian perez become famous      ---> Armando Pérez (born January 15, 1981), better known by his stage name Pitbull, is an American rapper, songwriter, and record producer.
what bird family is the owl                        ---> Owls are a group of birds that belong to the order Strigiformes, constituting 200 extant bird of prey species .
how many people were killed in the oklahoma city bombing ---> The Oklahoma blast claimed 168 lives, including 19 children under the age of 6, and injured more than 680 people.
Unique documents added: 26196
Loading model: Qwen/Qwen2-0.5B-Instruct ...


Using the latest cached version of the module from C:\Users\Diego\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--google_bleu\6fc70b7be0088120a372dfdd5d320b39b8bb3630cb8029b193941d9376e86bb0 (last modified on Fri Feb 13 17:36:42 2026) since it couldn't be found locally at evaluate-metric--google_bleu, or remotely on the Hugging Face Hub.


GLEU {'google_bleu': 0.023855755894590845}
LLM Judge Avg score(0-100): 57.91269841269841

####################################################################################################
Hyper-parameters: k = 7, temp = 0.1, prompt type = base
how big is bmc software in houston, tx             ---> 6,000
how long was i love lucy on the air                ---> six seasons
how did armando christian perez become famous      ---> his stage name Pitbull
what bird family is the owl                        ---> Strigiformes
how many people were killed in the oklahoma city bombing ---> 168
GLEU {'google_bleu': 0.052042801556420236}
LLM Judge Avg score(0-100): 58.34126984126984

####################################################################################################
Hyper-parameters: k = 7, temp = 0.1, prompt type = short
how big is bmc software in houston, tx             ---> 6,000
how long was i love lucy on the air                ---> a).
how did armando christian perez become 

Using the latest cached version of the module from C:\Users\Diego\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--google_bleu\6fc70b7be0088120a372dfdd5d320b39b8bb3630cb8029b193941d9376e86bb0 (last modified on Fri Feb 13 17:36:42 2026) since it couldn't be found locally at evaluate-metric--google_bleu, or remotely on the Hugging Face Hub.


GLEU {'google_bleu': 0.021325368157821618}
LLM Judge Avg score(0-100): 55.142857142857146

####################################################################################################
Hyper-parameters: k = 10, temp = 1.6, prompt type = short
how big is bmc software in houston, tx             ---> 1,135 megabytes
how long was i love lucy on the air                ---> 24
how did armando christian perez become famous      ---> is
what bird family is the owl                        ---> The relevant sentence in the passage is: o. " Owls are divided into two families
how many people were killed in the oklahoma city bombing ---> over
GLEU {'google_bleu': 0.01263187118267629}
LLM Judge Avg score(0-100): 57.03968253968254


### Test Set Behavior Analysis (Best Setup)

**Model:** flan-t5-small (LoRA: r=8, α=16, bias=none)  
**RAG Setup:** k=5, temperature=0.1, short prompt  
**Retriever:** MiniLM  
**LLM Judge Avg:** 62.36  
**GLEU:** 0.0493  



### Qualitative Behavior Analysis

#### 1️⃣ HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US  
**Prediction:** `0.8 to 0.9 million`  
- ❌ Incorrect and numerically fabricated.  
- The reference answer explains historical context (Atlantic slave trade).  
- The model outputs a **numeric estimate**, likely pulled from a related but mismatched context fragment.  
- This shows **retrieval grounding weakness** when multiple numeric passages exist.



#### 2️⃣ how a water pump works  
**Prediction:**  
`pump is a device that moves fluids ( liquids or gases )...`  
- ✅ Mostly correct.  
- The answer is concise and aligned with the definition.  
- Minor truncation (“sometimes s”) suggests generation length cutoff or decoding artifact.  
- Demonstrates good factual extraction from context.



#### 3️⃣ how old was sue lyon when she made lolita  
**Prediction:** `14`  
- ✅ Correct.  
- The reference states she was fourteen.  
- The model extracts the key fact cleanly and precisely.  
- Shows strong factoid QA behavior.



#### 4️⃣ how are antibodies used in  
**Prediction:**  
`Using this binding mechanism, an antibody can tag a microbe...`  
- ✅ Semantically correct.  
- Slightly paraphrased but conceptually aligned with immune system function.  
- Good contextual synthesis.



#### 5️⃣ HOW MUCH IS CENTAVOS IN MEXICO  
**Prediction:** `0`  
- ❌ Incorrect.  
- The correct answer is “100 centavos = 1 peso”.  
- The model incorrectly outputs a numeric token.  
- Likely retrieval confusion or numeric bias behavior.



### Quantitative Interpretation

- **LLM Judge Avg: 62.36**
  - Slightly higher than validation best (~61.15)
  - Indicates relatively stable generalization
- **GLEU: 0.049**
  - Moderate overlap
  - Again confirms that short factual answers reduce n-gram overlap with longer references



### Observed Behavioral Patterns

#### Strengths
- Strong at **short factual extraction** (numbers, names, definitions)
- Performs well at **direct definition-style questions**
- Stable under **low temperature (0.1)**

#### Weaknesses
- Sensitive to **numeric retrieval noise**
- Can hallucinate plausible but incorrect numbers
- Struggles when question requires **historical explanation rather than single fact extraction**
- Occasionally truncates answers



### Overall Assessment

On the test set, the model demonstrates:

- Good **factoid QA capability**
- Strong performance for short, direct answers
- Moderate robustness under deterministic decoding
- Some vulnerability to numeric hallucination

The LLMJudge score (~62.36) reflects that most answers are semantically acceptable, but occasional numeric or contextual errors prevent higher scoring.

Overall, this configuration remains **stable, efficient, and reasonably accurate** for a CPU-based RAG system, particularly for concise factual questions.


In [20]:
qa_grouped = question_answer_group(test_ds)
q_count = len(qa_grouped)
print('Reference QA:')
for i in range(5):
    sample = qa_grouped[i]
    print(f"{sample['question']:50} ---> {sample['correct_answer']}")
retriever = Retriever('MiniLM', test_device)
retriever.add_docs(all_docs)
judge = LLMJudge()
rank = 8
alpha = 16
bias = 'none'
k = 5
temp = 0.1
prompt_type = 'short'
generator = Generator('ft5', test_device, ft_model=True, rank=rank, alpha=alpha, bias=bias)
print(f'Testing{GENERATION_MODELS["ft5"][0]} fine-tunned with rank={rank}, alpha={alpha}, bias={bias}:')
print(f'Hyper-parameters: k = {k}, temp = {temp}, prompt type = {prompt_type}')
rag = RAG(retriever, generator, k, temp, prompt_type=prompt_type)
predictions, references = [], []
i = 0
for item in qa_grouped:
    predictions.append(rag.ask(item['question']))
    references.append(item['correct_answer'])
    if i < 5:
        print(f"{item['question']:50} ---> {predictions[-1]}")
    i += 1

gleu_result = evaluate_google_bleu(predictions, references)
print("GLEU", gleu_result)
sum_judge = 0
for i in range(len(references)):
    sum_judge += judge.judge(references[i], predictions[i])
print(f"LLM Judge Avg score(0-100): {sum_judge / len(references)}")

Reference QA:
HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US    ---> As such, African immigrants are to be distinguished from African American people, the latter of whom are descendants of mostly West and Central Africans who were involuntarily brought to the United States by means of the historic Atlantic slave trade .
how a water pump works                             ---> Pumps operate by some mechanism (typically reciprocating or rotary ), and consume energy to perform mechanical work by moving the fluid.
how old was sue lyon when she made lolita          ---> The actress who played Lolita, Sue Lyon , was fourteen at the time of filming.
how are antibodies used in                         ---> An antibody (Ab), also known as an immunoglobulin (Ig), is a large Y-shaped protein produced by B-cells that is used by the immune system to identify and neutralize foreign objects such as bacteria and viruses .
HOW MUCH IS CENTAVOS IN MEXICO                     ---> The peso is subdivided in

## 6. Conceptual Analysis

This homework explored the design, evaluation, and optimization of a CPU-based Retrieval-Augmented Generation (RAG) system using WikiQA, dense retrieval (MiniLM/E5), and small language models (flan-t5-small and variants). The following conceptual analysis synthesizes the empirical findings.


### 1. Impact of Embedding Models in RAG Architecture

In a RAG pipeline, the embedding model directly determines **retrieval quality**, which in turn constrains the upper bound of generation quality. The experiments clearly demonstrated that:

#### a. Embedding quality controls grounding
A strong embedding model (e.g., MiniLM) improves:
- Semantic similarity alignment
- Recall@k
- Precision@k

Since the generator only sees the retrieved documents, any retrieval failure propagates to generation. When embeddings retrieve noisy or partially relevant passages, the generator tends to:
- Extract incorrect numbers
- Produce semantically related but wrong facts
- Hallucinate under uncertainty

Thus, embedding performance defines the **information bottleneck** of the RAG system.

#### b. Speed–quality trade-off
The comparison between MiniLM and E5 revealed:
- Comparable retrieval accuracy
- Significant CPU runtime differences

Although E5 can be powerful, its indexing and query embedding time on CPU was substantially higher, without measurable gains in recall or precision in this setting. Therefore, embedding choice must balance:
- Retrieval accuracy
- Indexing cost
- Inference latency

In CPU-only deployment, efficient models like MiniLM achieve a better **accuracy-per-compute ratio**.

#### c. Retrieval noise sensitivity
Increasing `k` (number of retrieved documents) sometimes degraded performance. This highlights that embedding quality interacts with:
- Document ranking precision
- Noise tolerance of the generator

Higher `k` increases recall but can introduce irrelevant context, which confuses small generators.


### 2. Challenges and Limitations of Deploying RAG on CPU

Deploying RAG on CPU introduces both computational and modeling constraints.

#### a. Latency and throughput constraints
Observed limitations include:
- Slow embedding computation for large corpora
- Slower generation compared to GPU
- Training LoRA adapters requiring several minutes even for small models

CPU constraints limit:
- Model size
- Batch size
- Real-time scalability

This necessitates careful model selection and hyperparameter tuning.

#### b. Limited model capacity
Small language models (SLMs) such as flan-t5-small:
- Are efficient
- But have limited reasoning depth
- Are sensitive to prompt format
- Are prone to numeric hallucination

When fine-tuned narrowly (e.g., for classification), performance on open-ended QA slightly degraded because the base model was already instruction-tuned.

#### c. Hallucination under higher temperature
Higher decoding temperature significantly reduced stability:
- Numerical drift
- Fabricated statistics
- Context misinterpretation

On CPU, deterministic decoding (low temperature) proved necessary for reliability.

#### d. Retrieval-generation misalignment
Even with good embeddings, generation errors arose when:
- Retrieved passages contained multiple numeric facts
- Context was partially relevant
- Questions required explanatory synthesis rather than fact extraction

Small models struggle more with multi-hop reasoning and ambiguity.


### 3. Suggestions for Improving Retrieval and Generation

Based on experimental findings, several improvements can be proposed.

#### A. Retrieval Improvements

1. **Hybrid Retrieval (Dense + Sparse)**
   - Combine MiniLM dense embeddings with BM25 lexical matching.
   - Reduces semantic drift and improves precision.

2. **Re-ranking Stage**
   - Add a cross-encoder reranker after initial FAISS retrieval.
   - This can improve ranking quality without large architecture changes.

3. **Better Document Chunking**
   - Split documents into semantically coherent chunks.
   - Avoid mixing unrelated numeric information in the same passage.

4. **Dynamic k Selection**
   - Instead of fixed k, select k based on similarity score threshold.
   - Prevent unnecessary context noise.


#### B. Generation Improvements

1. **Prompt Engineering**
   - The “short” prompt outperformed the longer instruction-heavy prompt.
   - Cleaner prompts reduce instruction-token bias and improve grounding.

2. **Constrained Decoding**
   - For numeric answers, apply:
     - Token constraints
     - Output length limits
   - This can reduce numeric hallucinations.

3. **Answer Extraction Instead of Free Generation**
   - Switch from generative answering to span extraction from retrieved passages.
   - Especially effective for factoid QA.

4. **Lower Temperature Decoding**
   - Empirically, temperature = 0.1 provided the most stable results.
   - Deterministic decoding is recommended for factual QA.


### Overall Conceptual Conclusion

The homework demonstrates that in CPU-based RAG systems:

- **Embedding quality is foundational** and determines the ceiling of answer correctness.
- Efficient embeddings (MiniLM) provide optimal performance under CPU constraints.
- Small instruction-tuned models are surprisingly competitive when paired with strong retrieval.
- Fine-tuning must be carefully aligned with the downstream generation objective.
- Hyperparameters such as `k`, temperature, and prompt format significantly influence stability and hallucination behavior.

Ultimately, the best-performing configuration achieved a balanced trade-off between:
- Retrieval precision
- Deterministic generation
- Computational efficiency

This highlights a key principle in practical RAG design:  
> Optimization must consider the full retrieval–generation pipeline, not isolated components.
